In [ ]:
import clickhouse_connect
client = clickhouse_connect.get_client (
    host = "tp17.wb-bank.ru",
    port = 443,
    username = "holdzhgonov.a",
    secure = True,
    verify = False,
    client_cert = "/home/jovyan/tsh/clickhouse-prod.crt",
    client_cert_key = "/home/jovyan/tsh/clickhouse-prod.key"
)

In [ ]:
import datetime
from datetime import timedelta
import time
import pandas as pd

In [3]:
print(client.query('select 1').result_rows[0])

(1,)


In [4]:
SCHEMA = 'sandbox'
TABLE = 'adhoc_cft_r2_vid_oper'

In [5]:
df1 = pd.read_csv('z#r2_vid_oper.csv', sep = ';', encoding='cp1251')
df1.columns = df1.columns.str.strip()
# df1 = df1[df1['status']=='OPEN']
# df1['rid'] = df1['srid']
# df1.sort_values(by="update_dt")
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1767 entries, 0 to 1766
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0                       1767 non-null   int64  
 1   ID                  1767 non-null   int64  
 2   SN                  1767 non-null   int64  
 3   SU                  1767 non-null   int64  
 4   C_NAME              1767 non-null   object 
 5   C_CODE              1767 non-null   object 
 6   C_BUS_PROCESS       1374 non-null   float64
 7   C_CODE_GROUP        1767 non-null   object 
 8   C_DEPEND_PLAN_OPER  541 non-null    float64
 9   C_SPOSOB_KVIT#0     1767 non-null   int64  
 10  C_CORR_VID_OPER     490 non-null    float64
 11  C_IS_GRACE          304 non-null    float64
dtypes: float64(4), int64(5), object(3)
memory usage: 165.8+ KB


In [6]:
df1 = df1[[ 'ID', 'SN', 'SU', 'C_NAME', 'C_CODE', 'C_BUS_PROCESS',
       'C_CODE_GROUP', 'C_DEPEND_PLAN_OPER', 'C_SPOSOB_KVIT#0',
       'C_CORR_VID_OPER', 'C_IS_GRACE']]
df1

,ID,SN,SU,C_NAME,C_CODE,C_BUS_PROCESS,C_CODE_GROUP,C_DEPEND_PLAN_OPER,C_SPOSOB_KVIT#0,C_CORR_VID_OPER,C_IS_GRACE
0,29397544,4,74553392,Гашение Задолженность перед покупателем,ГАШЕНИЕ_SALE_DEBT,29403237.0,R2_LOAN,NaN,4,NaN,NaN
1,29397545,4,74553392,Гашение ВНБ. Просроченные проценты за просроче...,ГАШЕНИЕ_ВНБ-ПРОЦ112ПРОСРОЧ-VN,29403237.0,R2_LOAN,NaN,1,NaN,NaN
2,29397546,4,74553392,Гашение ВНБ. Просроченных процентов (учт на вн...,ГАШЕНИЕ_ВНБ-ПРОЦПРОСРОЧ-VN,29403237.0,R2_LOAN,NaN,1,NaN,NaN
3,29397547,4,74553392,Гашение ВНБ. Плата за обслуживание карты Проср...,ГАШЕНИЕ_ВНБ_DIS-COMCARD-PR-BAL,29403237.0,R2_LOAN,29397583.0,3,NaN,NaN
4,29397548,4,74553392,Взимание НДС (ВНБ. Просроченная комиссия за из...,ГАШЕНИЕ_ВНБ_NDS_POS_458,29403237.0,R2_LOAN,29397472.0,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1762,10287278294,1,390027093,"ВББ. Перенос. Увеличение резерва ""Просроченная...",WBB_TRANSF_INC_OVERDUE_LOAN,29403483.0,R2_RES_PORT,NaN,4,NaN,0.0
1763,10287278296,1,390027093,"ВББ. Перенос. Увеличение резерва ""Просроченные...",WBB_TRANSF_INC_OVERDUE_INTEREST,29403483.0,R2_RES_PORT,NaN,4,NaN,0.0
1764,10287278298,1,390027093,"ВББ. Перенос. Увеличение резерва ""Просроченная...",WBB_TRANSF_INC_OVERDUE_CRED_COM,29403483.0,R2_RES_PORT,NaN,4,NaN,0.0
1765,10935177216,1,396317608,ВББ. Увеличение лимита в процессинге,WBB_LIMIT_BALANCE_INCREASE,29403267.0,R2_LOAN,NaN,4,NaN,NaN


In [7]:
import pandas as pd

SCHEMA = 'sandbox'
TABLE = 'adhoc_cft_r2_vid_oper'

columns = [
    'ID',
    'SN',
    'SU',
    'C_NAME',
    'C_CODE',
    'C_BUS_PROCESS',
    'C_CODE_GROUP',
    'C_DEPEND_PLAN_OPER',
    'C_SPOSOB_KVIT_0',
    'C_CORR_VID_OPER',
    'C_IS_GRACE',
]

df_output = df1.copy()

df_output = df_output.rename(columns={
    'C_SPOSOB_KVIT#0': 'C_SPOSOB_KVIT_0'
})

df_output = df_output[columns]

int_cols = [
    'ID',
    'SN',
    'SU',
    'C_SPOSOB_KVIT_0',
]

nullable_int_cols = [
    'C_BUS_PROCESS',
    'C_DEPEND_PLAN_OPER',
    'C_CORR_VID_OPER',
    'C_IS_GRACE',
]

str_cols = [
    'C_NAME',
    'C_CODE',
    'C_CODE_GROUP',
]

for col in int_cols:
    df_output[col] = pd.to_numeric(df_output[col], errors='raise').astype('int64')

for col in nullable_int_cols:
    df_output[col] = pd.Series(
        [None if pd.isna(x) else int(x) for x in df_output[col]],
        dtype='object'
    )

for col in str_cols:
    df_output[col] = pd.Series(
        [None if pd.isna(x) else str(x) for x in df_output[col]],
        dtype='object'
    )


In [8]:
print(df_output.dtypes)

for col in nullable_int_cols:
    print(col, df_output[col].map(lambda x: type(x).__name__).value_counts(dropna=False))


ID                     int64
SN                     int64
SU                     int64
C_NAME                object
C_CODE                object
C_BUS_PROCESS         object
C_CODE_GROUP          object
C_DEPEND_PLAN_OPER    object
C_SPOSOB_KVIT_0        int64
C_CORR_VID_OPER       object
C_IS_GRACE            object
dtype: object
C_BUS_PROCESS C_BUS_PROCESS
int         1374
NoneType     393
Name: count, dtype: int64
C_DEPEND_PLAN_OPER C_DEPEND_PLAN_OPER
NoneType    1226
int          541
Name: count, dtype: int64
C_CORR_VID_OPER C_CORR_VID_OPER
NoneType    1277
int          490
Name: count, dtype: int64
C_IS_GRACE C_IS_GRACE
NoneType    1463
int          304
Name: count, dtype: int64


In [9]:
res = client.query("""
SELECT
    hostName(),
    database,
    table,
    is_readonly,
    is_session_expired,
    future_parts,
    queue_size,
    absolute_delay,
    zookeeper_exception
FROM clusterAllReplicas('clickhouse', system.replicas)
WHERE database = 'sandbox'
  AND table = 'adhoc_cft_r2_vid_oper_local'
ORDER BY hostName()
""")

for row in res.result_rows:
    print(row)


('chi-clickhouse-prod-clickhouse-0-0-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')
('chi-clickhouse-prod-clickhouse-0-1-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')
('chi-clickhouse-prod-clickhouse-1-0-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')
('chi-clickhouse-prod-clickhouse-1-1-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')
('chi-clickhouse-prod-clickhouse-2-0-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')
('chi-clickhouse-prod-clickhouse-2-1-0', 'sandbox', 'adhoc_cft_r2_vid_oper_local', 0, 0, 0, 0, 0, '')


In [10]:
import json
import numpy as np

def clean_value(x):
    if pd.isna(x):
        return None
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        if float(x).is_integer():
            return int(x)
        return float(x)
    return x

records = []

for row in df_output[columns].to_dict(orient='records'):
    clean_row = {
        col: clean_value(row[col])
        for col in columns
    }
    records.append(clean_row)

payload = '\n'.join(
    json.dumps(r, ensure_ascii=False, allow_nan=False)
    for r in records
).encode('utf-8')

client.raw_insert(
    table='sandbox.adhoc_cft_r2_vid_oper',
    column_names=columns,
    insert_block=payload,
    fmt='JSONEachRow'
)
